# 02 — Register map and driver

This notebook is mostly informational: it walks through the
AXI4-Lite register map, decodes a STATUS word, and shows the
driver's hot path. The actual `MicroGPT` driver requires PYNQ +
the bitstream loaded on a real PYNQ-Z2 board, so the generate()
calls are guarded — they will work on the board, not on a dev
laptop.


## STATUS register decoder

The STATUS register at offset `0x00C` packs several fields into a
single 32-bit word. The exact layout lives in
`hw/src/top/microgpt_pynq_top.sv`. Here is a small decoder you can
run on the raw u32 value to see the meaning at a glance.


In [ ]:
def decode_status(u32):
    return {
        'ready':       bool(u32 & (1 << 0)),
        'busy':        bool(u32 & (1 << 1)),
        'done':        bool(u32 & (1 << 2)),
        'error':       bool(u32 & (1 << 3)),
        'toggle':      bool(u32 & (1 << 4)),
        'direct_mode': bool(u32 & (1 << 5)),
        'out_len':     (u32 >> 16) & 0xFF,
        'pos':         (u32 >> 24) & 0xFF,
    }

# Example: an idle, post-reset status
decode_status(0x0000_0001)


In [ ]:
# Example: 'busy and generating, position 3, no errors yet'
decode_status(0x0300_0002)


## Driver hot path

The deployed driver caches a `uint32` view over the MMIO array once
per instance, then does direct burst reads. Here is the relevant
snippet from `sw/drivers/microgpt.py`:

```python
self._u32 = np.asarray(self.mmio.array, dtype=np.uint32)
...
# tight wait loop, time-check every 4096 spins (not every spin)
spins = 0
while True:
    if self._u32[A_STATUS >> 2] & (1 << ST_DONE_BIT):
        break
    spins += 1
    if spins & 0xFFF == 0 and time.perf_counter() - t0 > self._timeout_s:
        raise TimeoutError('done bit never set')

# burst-read 16 generated tokens with one numpy view
tokens = (self._u32[A_OUTPUT_MEM >> 2 : (A_OUTPUT_MEM >> 2) + n] & 0xFF).tolist()
```

The `mmio.array` view + masking pattern is roughly **1.7× faster**
than the obvious `[mmio.read(addr + i*4) for i in range(n)]` loop
because each `mmio.read()` does a fresh attribute lookup and a
Python-level branch. See `sw/notebooks/throughput.ipynb` for the
measurement on the deployed board.


## IRQ fast path

An optional `use_irq=True` flag opens `/dev/uio<n>` (PL fabric IRQ)
and replaces the spin loop with a blocking `os.read`:

```python
import os, struct
self._uio_fd = os.open(f'/dev/uio{fabric_uio_index}', os.O_RDWR)
...
# enable the IRQ once
os.write(self._uio_fd, struct.pack('<I', 1))
# block until the PL pulses done_irq
_ = os.read(self._uio_fd, 4)
```

On the deployed board this lands the same latency without burning
a core in the spin loop. The UIO index discovery is the small
`_open_fabric_uio()` helper in the driver (it walks
`/sys/class/uio/*/maps/map0/name` looking for a fabric match).


## Generate tokens (board-only)

If you are running this notebook on the PYNQ-Z2 with the overlay
deployed under `/home/xilinx/jupyter_notebooks/microgpt/`, the
cell below will produce text. On a dev laptop it will raise
(`pynq` is not importable). That is expected.


In [ ]:
import sys
from pathlib import Path
drivers = (Path('../sw/drivers')).resolve()
if str(drivers) not in sys.path:
    sys.path.insert(0, str(drivers))

try:
    from microgpt import MicroGPT
    gpt = MicroGPT()
    text, info = gpt.generate(max_tokens=8, temperature=1.0, seed=42)
    print(f'text   = {text!r}')
    print(f'cycles = {info["cycles"]}')
    print(f'tokens = {info["tokens"]}')
except ImportError as e:
    print('Skipped (board-only):', e)
except Exception as e:
    print('Skipped (no overlay loaded):', type(e).__name__, e)
